# 模块4：LightGBM模型训练教学
---
## 🎯 学习目标
学完本模块你将能够：
- 理解LightGBM模型的原理和优势
- 掌握LightGBM全部核心参数的含义和调优方法
- 学会样本不平衡处理、交叉验证、早停机制的原理和实现
- 能够独立完成二分类模型的训练、评估和调优全流程
- 掌握特征重要性分析方法，能够反向指导特征工程

---
## 📚 知识点讲解：LightGBM模型基础

### 1. 什么是LightGBM？
LightGBM（Light Gradient Boosting Machine）是微软亚洲研究院2017年推出的梯度提升树框架，是目前结构化数据建模的首选算法，在数据挖掘比赛中获奖率超过90%。

### 2. LightGBM的核心优势
| 优势 | 说明 |
|------|------|
| 速度快 | 采用直方图算法和单边梯度采样，训练速度是XGBoost的5-10倍 |
| 内存小 | 直方图算法将连续特征离散化，内存占用仅为XGBoost的1/10 |
| 精度高 | 采用 leaf-wise 生长策略，比 level-wise 精度更高，不容易过拟合 |
| 支持类别特征 | 原生支持类别特征，不需要手动做One-Hot编码 |
| 支持并行计算 | 支持特征并行、数据并行、投票并行，适合大数据集 |

### 3. 梯度提升树原理
梯度提升树是集成学习的一种，通过多棵决策树的预测结果累加得到最终预测值：
1. 第一棵树学习原始数据的规律
2. 第二棵树学习第一棵树的预测误差
3. 第三棵树学习前两棵树的预测误差
4. 以此类推，所有树的预测结果相加就是最终结果

每次迭代都沿着梯度下降的方向减小误差，所以叫梯度提升。

---
## 💻 代码逐行拆解：数据加载与预处理


In [ ]:
# 1. 导入需要的库
import pandas as pd
import numpy as np
import lightgbm as lgb
import pickle
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

# 绘图配置
%matplotlib inline
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 2. 加载特征工程完成的数据集
df = pd.read_csv("final_train_dataset_fixed.csv")  # 上一模块输出的数据集

print(f"原始数据集形状: {df.shape}")
print(f"原始正负样本比例: 1:{len(df[df.label == 0])/len(df[df.label == 1]):.1f}")

### 样本不平衡处理
我们的数据集中正负样本比例约为1:100，属于极端不平衡情况。有三种常用的处理方法：
1. **欠采样**：丢弃大部分负样本，让正负比例接近1:10或1:20
2. **过采样**：复制正样本，或者用SMOTE算法生成新的正样本
3. **类别权重**：给正样本更大的损失权重，让模型更关注正样本

本项目我们采用**欠采样+类别权重**的组合方案，效果最好。

In [ ]:
# 欠采样：保留所有正样本，随机采样20倍的负样本
df_pos = df[df['label'] == 1]  # 正样本
df_neg = df[df['label'] == 0].sample(n=len(df_pos)*20, random_state=42)  # 负样本采样20倍
df_sampled = pd.concat([df_pos, df_neg], ignore_index=True)

print(f"采样后数据集形状: {df_sampled.shape}")
print(f"采样后正负样本比例: 1:{len(df_neg)/len(df_pos):.1f}")

**为什么采样比例是1:20？**
- 1:1比例会丢失太多负样本信息，模型容易过拟合
- 1:50比例不平衡程度还是太高，模型学不到正样本的规律
- 1:10~1:30是工业界常用的比例，既能保留足够的负样本信息，又能让模型充分学习正样本的规律

In [ ]:
# 分离特征和标签
exclude_cols = ["user_id", "item_id", "label"]
feature_cols = [col for col in df_sampled.columns if col not in exclude_cols]

X = df_sampled[feature_cols]
y = df_sampled["label"]

# 划分训练集和测试集（8:2划分，分层抽样保证正负比例不变）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集大小: {len(X_train):,}, 测试集大小: {len(X_test):,}")
print(f"特征数量: {len(feature_cols)}")

---
## 💻 代码逐行拆解：LightGBM参数详解


In [ ]:
# LightGBM核心参数配置
params = {
    # 基础参数
    "boosting_type": "gbdt",        # 提升树类型：gbdt（传统梯度提升树）、dart（带dropout的提升树）、goss（梯度单边采样）
    "objective": "binary",          # 任务类型：binary二分类、regression回归、multiclass多分类
    "metric": "auc",                # 评估指标：auc（二分类首选）、binary_logloss（对数损失）、f1
    
    # 训练参数
    "learning_rate": 0.05,          # 学习率：步长，越小训练越慢，泛化性越好，常用0.01~0.1
    "num_leaves": 63,               # 每棵树的叶子节点数：最多2^max_depth - 1，太大容易过拟合
    "max_depth": 7,                 # 树的最大深度：限制树的生长，防止过拟合，常用3~10
    "min_child_samples": 100,       # 叶子节点最少样本数：小于这个值就不再分裂，防止过拟合，常用10~200
    
    # 采样参数（防止过拟合）
    "subsample": 0.8,               # 行采样：每棵树随机使用80%的样本
    "colsample_bytree": 0.8,        # 列采样：每棵树随机使用80%的特征
    "reg_alpha": 0.1,               # L1正则化：惩罚大的权重，防止过拟合
    "reg_lambda": 0.1,              # L2正则化：惩罚大的权重，防止过拟合
    
    # 不平衡样本参数
    "scale_pos_weight": 20,         # 正样本权重：等于负样本/正样本比例，平衡正负样本的损失
    
    # 其他参数
    "random_state": 42,             # 随机种子，保证结果可复现
    "verbosity": -1,                # 日志级别：-1不输出日志，0输出错误，1输出信息
}

print("模型参数配置完成！")

### 参数调优优先级
1. **高优先级**：`learning_rate`、`num_leaves`、`max_depth`、`min_child_samples`
2. **中优先级**：`subsample`、`colsample_bytree`、`reg_alpha`、`reg_lambda`
3. **低优先级**：其他参数

💡 调优技巧：先固定学习率为0.05~0.1，调整树结构参数到最优，再降低学习率，增加迭代次数，获得更好的泛化性能。

---
## 💻 代码逐行拆解：交叉验证与早停机制

### 1. K折交叉验证
将数据集分成K份，每次用K-1份训练，1份验证，重复K次，取平均结果，避免单次划分的随机性，评估结果更可靠。

### 2. 早停机制
当验证集的评估指标连续N轮没有提升时，停止训练，防止过拟合，同时得到最优的迭代次数。

In [ ]:
# 5折分层交叉验证（保持每折的正负样本比例和整体一致）
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = []  # 保存每折的模型
auc_scores = []  # 保存每折的AUC分数

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n========== 第 {fold+1}/5 折训练 ==========")
    
    # 划分当前折的训练集和验证集
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 构建LightGBM数据集
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
    
    # 训练模型，带早停
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=10000,  # 最大迭代次数
        valid_sets=[lgb_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),  # 连续100轮没提升就停止
            lgb.log_evaluation(period=100),  # 每100轮输出一次日志
        ]
    )
    
    # 评估当前折模型
    y_val_pred = model.predict(X_val, num_iteration=model.best_iteration)
    val_auc = roc_auc_score(y_val, y_val_pred)
    auc_scores.append(val_auc)
    models.append(model)
    
    print(f"第 {fold+1} 折验证集AUC: {val_auc:.4f}, 最优迭代次数: {model.best_iteration}")

print(f"\n========== 交叉验证完成 ==========")
print(f"5折平均AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

**结果解读：**
- 平均AUC在0.9以上说明模型效果很好，AUC=0.9表示模型有90%的概率能正确区分一个正样本和一个负样本
- 标准差很小说明模型稳定性很好，不同折的结果差异不大
- 最优迭代次数通常在几百到几千之间，说明我们的参数设置比较合理

---
## 💻 代码逐行拆解：模型评估与分析

评估模型效果是建模过程中非常重要的一步，我们需要从多个维度评估模型的表现。

In [ ]:
# 选择AUC最高的模型作为最优模型
best_model = models[np.argmax(auc_scores)]

# 在测试集上预测
y_pred_proba = best_model.predict(X_test, num_iteration=best_model.best_iteration)
y_pred = (y_pred_proba >= 0.5).astype(int)  # 概率>=0.5预测为正样本，阈值可以调整

# 计算各项评估指标
test_auc = roc_auc_score(y_test, y_pred_proba)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

print("="*50)
print("测试集评估结果:")
print(f"AUC: {test_auc:.4f}")
print(f"召回率: {test_recall:.4f}")
print(f"F1-score: {test_f1:.4f}")
print("="*50)

### 评估指标说明
| 指标 | 公式 | 业务意义 |
|------|------|----------|
| AUC | 正样本预测得分高于负样本的概率 | 模型整体的区分能力，越高越好 |
| Recall（召回率）| TP/(TP+FN) | 实际购买的用户中，被我们正确预测出来的比例，越高越好 |
| Precision（精确率）| TP/(TP+FP) | 预测为购买的用户中，实际真的购买的比例，越高越好 |
| F1-score | 2*P*R/(P+R) | 精确率和召回率的调和平均，综合衡量模型效果 |

💡 比赛中我们的目标是最大化F1-score，可以通过调整分类阈值来平衡精确率和召回率。

In [ ]:
# 混淆矩阵可视化
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["预测未购买", "预测购买"],
    yticklabels=["实际未购买", "实际购买"],
)
plt.xlabel("预测标签")
plt.ylabel("真实标签")
plt.title("混淆矩阵")
plt.show()

# 分类报告
print("\n分类报告:")
print(classification_report(y_test, y_pred))

---
## 💻 代码逐行拆解：特征重要性分析

特征重要性分析可以帮助我们理解哪些特征对预测结果影响最大，反向指导特征工程优化。

In [ ]:
# 获取特征重要性（importance_type='gain'表示特征带来的总增益）
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": best_model.feature_importance(importance_type="gain"),
})
feature_importance = feature_importance.sort_values("importance", ascending=False).reset_index(drop=True)

# 可视化Top20重要特征
plt.figure(figsize=(10, 8))
sns.barplot(x="importance", y="feature", data=feature_importance.head(20), palette="viridis")
plt.title("Top20 特征重要性", fontsize=14)
plt.xlabel("重要性得分", fontsize=12)
plt.ylabel("特征名", fontsize=12)
plt.tight_layout()
plt.show()

print("Top10 核心特征:")
print(feature_importance.head(10).to_string(index=False))

**结果解读：**
- 通常交互类特征（用户对商品的加购、购买次数等）会排在最前面，这些是强特
- 时间衰减特征的重要性通常也很高，说明近期行为确实更有预测价值
- 如果发现某个特征重要性为0，说明这个特征没有任何预测价值，可以删除，减少特征维度
- 可以针对重要性高的特征做进一步衍生，提升模型效果


---
## 💻 代码逐行拆解：模型保存与部署


In [ ]:
# 保存最优模型
with open("lgb_repurchase_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# 加载模型的方法
with open("lgb_repurchase_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("模型保存和加载测试完成！")

---
## 🎯 实战例题

请你独立完成以下练习：
1. 尝试调整分类阈值，找到F1-score最高的阈值
2. 调整模型参数，尝试将AUC提升0.01以上
3. 尝试删除重要性最低的5个特征，重新训练模型，看效果是否下降
4. 尝试用全部数据（不欠采样）训练模型，对比欠采样的效果差异


---
## ✅ 小测验

### 选择题
1. LightGBM中哪个参数用来处理样本不平衡问题？
   A. learning_rate
   B. scale_pos_weight
   C. num_leaves
   D. subsample

2. 早停机制的作用是？
   A. 加快训练速度
   B. 防止过拟合
   C. 提高模型精度
   D. 减少内存占用

3. 以下哪个评估指标最适合衡量不平衡样本的模型效果？
   A. 准确率
   B. AUC
   C. 召回率
   D. F1-score

### 简答题
1. 为什么我们要使用交叉验证而不是单次划分训练集和测试集？
2. 欠采样有什么优点和缺点？

---
## 📝 重点总结

本模块核心知识点：
1. **LightGBM优势**：速度快、内存小、精度高、支持类别特征，是结构化数据建模的首选
2. **核心参数**：学习率、树的深度/叶子节点数、正则化参数、不平衡样本参数
3. **不平衡样本处理**：欠采样、过采样、类别权重三种方法的适用场景
4. **模型训练流程**：K折交叉验证 + 早停机制，保证模型泛化能力
5. **模型评估**：AUC、召回率、精确率、F1-score等指标的业务含义
6. **特征重要性**：理解特征重要性，反向指导特征工程优化

下一个模块我们将学习如何使用训练好的模型生成比赛提交结果，以及如何优化结果提升比赛排名。